# 17 · Reproduce paper tables, figures and evidence index

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

This notebook regenerates outputs from saved results. It does not invent missing experiments or write a results section from the plan.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Verify the analysis lock and actual output inventory

In [ ]:
from oncoplate.governance import verify_lock
from oncoplate.reporting import reproduce_index
lock=verify_lock(p['private']/'analysis_lock.json')
index=reproduce_index(p['reports'],p['reports']/'reproduce_index.json')
print(index)

## 2. Recompute the main table from protected predictions and ratings

In [ ]:
from oncoplate.evaluation import evaluate_results
results=read_table(p['private']/'test_evaluation_results.csv');results['seed']=results.seed.astype(int)
table=evaluate_results(results,lock['protocol']['domain_mixture'])
write_table(p['reports']/'paper_main_table.csv',table)
print(table.to_markdown(index=False))

## 3. Produce a completion checklist without filling missing results

In [ ]:
required=['main_selective_results.csv','primary_paired_comparison.json','primary_risk_coverage.png','external_selective_results.csv','clarification_test_summary.csv']
status=[{'artifact':x,'present':(p['reports']/x).exists()} for x in required]
write_json(p['reports']/'paper_evidence_completeness.json',status)
print(json.dumps(status,indent=2))
print((REPO/'docs/PAPER_REPORTING.md').read_text())

## 4. Create a deliberately separate aggregate-release directory

In [ ]:
release=p['root']/'release_review';release.mkdir(exist_ok=True)
print('No raw data or private prediction tables have been copied. Review actual permissions and redaction before a release.')
print('Use notebook 99 for manual Git review. A paper draft must distinguish completed, exploratory and missing experiments.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
